In [ ]:
import cv2
import numpy as np
import ipywidgets as widgets
from IPython.display import display
import threading
import queue
import mediapipe as mp
import time

In [ ]:
BaseOptions = mp.tasks.BaseOptions
PoseLandmarker = mp.tasks.vision.PoseLandmarker
PoseLandmarkerOptions = mp.tasks.vision.PoseLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

MODEL_PATH = "pose_landmarker_full.task"


POSE_CONNECTIONS = [

    # TWARZ
    (0, 1), (1, 2), (2, 3), (3, 7),
    (0, 4), (4, 5), (5, 6), (6, 8),

    (9, 10),

    # TUŁÓW
    (11, 12),
    (11, 23),
    (12, 24),
    (23, 24),

    # LEWA RĘKA
    (11, 13),
    (13, 15),

    (15, 17),  # mały palec
    (15, 19),  # wskazujący
    (15, 21),  # kciuk

    (17, 19),

    # PRAWA RĘKA
    (12, 14),
    (14, 16),

    (16, 18),
    (16, 20),
    (16, 22),

    (18, 20),

    # LEWA NOGA
    (23, 25),
    (25, 27),

    (27, 29),  # pięta
    (29, 31),  # czubek stopy

    # PRAWA NOGA
    (24, 26),
    (26, 28),

    (28, 30),
    (30, 32),

    # DODATKOWE POŁĄCZENIA STOPY
    (27, 31),
    (28, 32),
]


def draw_pose(frame, landmarks):
    h, w = frame.shape[:2]


    for start_idx, end_idx in POSE_CONNECTIONS:
        p1 = landmarks[start_idx]
        p2 = landmarks[end_idx]

        x1 = int(p1.x * w)
        y1 = int(p1.y * h)

        x2 = int(p2.x * w)
        y2 = int(p2.y * h)

        cv2.line(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)


    for lm in landmarks:
        x = int(lm.x * w)
        y = int(lm.y * h)

        cv2.circle(frame, (x, y), 4, (0, 0, 255), -1)


options = PoseLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=MODEL_PATH),
    running_mode=VisionRunningMode.VIDEO,
    num_poses=1,
)


In [ ]:
# Do przekazywania obrazu używałem na telefonie aplikacji IP webcam na androidzie
STREAM_URL = 0 #"http://192.168.135.9:8080/video"

MAX_WIDTH = 900
JPEG_QUALITY = 50
FPS = 15
TIMEOUT = 200 # zatrzymuje kamerke po X sekundach

# TODO:
# implementacja tego ale zamiast łączenia z kamerką to danie po prostu scierzki do video
# analiza obrazu wg wymagań projektu
# gl hf


In [12]:

stop = threading.Event()
q = queue.Queue(maxsize=1)
img_widget = widgets.Image(format='jpeg')


def resize(frame):
	h, w = frame.shape[:2]
	if w > MAX_WIDTH:
		return cv2.resize(frame, (MAX_WIDTH, int(h * MAX_WIDTH / w)))
	return frame


def analyze(frame):
	frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
	frame = cv2.cvtColor(frame, cv2.COLOR_GRAY2BGR)
	return frame


def fetch():
	# cap = cv2.VideoCapture(STREAM_URL)
	# cap.set(cv2.CAP_PROP_FPS, FPS)
	# while not stop.is_set():
	# 	ret, frame = cap.read()
	# 	if ret:
	# 		try:
	# 			q.put_nowait(resize(frame))
	# 		except queue.Full:
	# 			pass
	# cap.release()
    cap = cv2.VideoCapture(STREAM_URL)
    cap.set(cv2.CAP_PROP_FPS, FPS)
    with PoseLandmarker.create_from_options(options) as landmarker:

        while cap.isOpened() and not stop.is_set():
            ret, frame = cap.read()
            if not ret:
                break

            frame = cv2.flip(frame, 1)

            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

            mp_image = mp.Image(
                image_format=mp.ImageFormat.SRGB,
                data=rgb
			)

            timestamp_ms = int(time.time() * 1000)

            result = landmarker.detect_for_video(
                mp_image,
                timestamp_ms
            )

            if result.pose_landmarks:
                draw_pose(frame, result.pose_landmarks[0])

            try:
                q.put_nowait(resize(frame))
            except queue.Full:
                pass

            if cv2.waitKey(1) & 0xFF == 27:  # ESC
                break
    cap.release()
    cv2.destroyAllWindows()


def process():
	while not stop.is_set():
		try:
			frame = q.get(timeout=1)
			frame = analyze(frame)
			_, buf = cv2.imencode('.jpg', frame, [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY])
			img_widget.value = buf.tobytes()
		except:
			pass


def start():
	display(img_widget)
	threading.Thread(target=fetch, daemon=True).start()
	threading.Thread(target=process, daemon=True).start()
	threading.Timer(TIMEOUT, stop.set).start()

start()


Image(value=b'', format='jpeg')

In [13]:
stop.set()